## Transformaciones: artefactos y su uso

Se construyen los tres artefactos que alimentan las Partes II–VI (**vectores de características**, **matrices de
interacción** y **grafos**), se explica para qué sirve cada uno y cómo el particionamiento afecta el plan de ejecución

- **Corre en:** Spark. Los self-joins de co-participación generan millones de pares; las técnicas O(n²) de las partes
  siguientes trabajan luego en local sobre estos artefactos ya reducidos

### Utilidades

In [1]:
NOTEBOOK = "c_uso"
%run ../funciones.ipynb

In [2]:
spark = crear_spark(NOTEBOOK)
verificar_cluster(spark)

Master        : spark://spark-master:7077
Executors     : 2
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


### Carga

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

hechos = leer(final_dir, "hechos").where(F.col("anio").isin(anios))
proc   = leer(final_dir, "procesos").where(F.col("anio").isin(anios))
part   = leer(final_dir, "participantes").where(F.col("anio").isin(anios))
dim_e, dim_p = leer(final_dir, "entidades"), leer(final_dir, "proveedores")

# nivel award × proveedor: deshace la explosión por familia
hp = (hechos.groupBy("ocid", "award_id", "proveedor_key", "entidad_key", "region", "categoria",
                     "postor_unico", "es_directa", "anio", "mes")
            .agg(F.sum("monto_asignado").alias("monto"))).cache()
print(f"award × proveedor: {hp.count():,}")

award × proveedor: 199,377


---
### 1. Vectores de características

#### a. Proveedor

In [4]:
def dependencia(df, actor, contraparte, sufijo):
    # share del principal cliente/proveedor y HHI de la distribución del monto
    w = Window.partitionBy(actor)
    return (df.groupBy(actor, contraparte).agg(F.sum("monto").alias("m"))
              .withColumn("s", F.col("m") / F.sum("m").over(w))
              .groupBy(actor).agg(F.max("s").alias(f"share_top_{sufijo}"), F.sum(F.col("s") ** 2).alias(f"hhi_{sufijo}"),
                                  F.count("*").alias(f"n_{sufijo}")))

cat = lambda v: F.avg((F.col("categoria") == v).cast("int"))
n_familias = lambda k: hechos.where("familia <> 'SIN'").groupBy(k).agg(F.countDistinct("familia").alias("n_familias"))

In [5]:
ofe_p = part.groupBy("proveedor_key").agg(F.sum("en_lista_oferentes").alias("n_ofertas"),
                                          F.sum(F.col("en_lista_oferentes") * F.col("gano")).alias("n_ganadas_compitiendo"))
feat_prov = (hp.groupBy("proveedor_key").agg(
        F.countDistinct("ocid").alias("n_procesos"), F.countDistinct("region").alias("n_regiones"),
        F.sum("monto").alias("monto_total"), F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano"),
        F.avg("postor_unico").alias("pct_postor_unico"), F.avg("es_directa").alias("pct_directa"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"),
        F.countDistinct("anio", "mes").alias("meses_activos"))
    .join(dependencia(hp, "proveedor_key", "entidad_key", "entidades"), "proveedor_key", "left")
    .join(n_familias("proveedor_key"), "proveedor_key", "left").join(ofe_p, "proveedor_key", "left")
    .join(dim_p.select("proveedor_key", "es_consorcio"), "proveedor_key", "left")
    .withColumn("compite", (F.coalesce("n_ofertas", F.lit(0)) > 0).cast("int"))
    .withColumn("tasa_exito", F.when(F.col("n_ofertas") > 0, F.col("n_ganadas_compitiendo") / F.col("n_ofertas")))
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

In [6]:
FEAT_PROV = ["log_n_procesos", "n_entidades", "n_regiones", "log_monto_total", "log_monto_mediano", "pct_postor_unico",
             "pct_directa", "pct_bienes", "pct_servicios", "pct_obras", "meses_activos", "share_top_entidades",
             "hhi_entidades", "n_familias", "compite", "tasa_exito", "es_consorcio"]
feat_prov = VectorAssembler(inputCols=FEAT_PROV, outputCol="features_raw").transform(feat_prov.fillna(0, FEAT_PROV))

#### b. Entidad

Las métricas de proceso salen de **todos** sus procesos, adjudicados o no

In [7]:
adjudicados = hp.select("ocid").distinct().withColumn("adjudicado", F.lit(1))
feat_ent = (proc.join(adjudicados, "ocid", "left").fillna({"adjudicado": 0})
    .groupBy("entidad_key").agg(
        F.count("*").alias("n_procesos"), F.avg("postor_unico").alias("pct_postor_unico"),
        F.avg("es_directa").alias("pct_directa"), F.avg("n_postores").alias("postores_promedio"),
        F.avg(1 - F.col("adjudicado")).alias("pct_sin_adjudicar"),
        cat("GOODS").alias("pct_bienes"), cat("SERVICES").alias("pct_servicios"), cat("WORKS").alias("pct_obras"))
    .join(hp.groupBy("entidad_key").agg(F.sum("monto").alias("monto_adjudicado_total"),
                                        F.expr("percentile_approx(monto, 0.5)").alias("monto_mediano")), "entidad_key", "left")
    .join(dependencia(hp, "entidad_key", "proveedor_key", "proveedores"), "entidad_key", "left")
    .join(n_familias("entidad_key"), "entidad_key", "left")
    .join(dim_e.select("entidad_key", "region"), "entidad_key", "left")
    .withColumn("log_n_procesos", F.log1p("n_procesos"))
    .withColumn("log_monto_total", F.log1p("monto_adjudicado_total")).withColumn("log_monto_mediano", F.log1p("monto_mediano")))

FEAT_ENT = ["log_n_procesos", "pct_postor_unico", "pct_directa", "postores_promedio", "pct_sin_adjudicar", "pct_bienes",
            "pct_servicios", "pct_obras", "log_monto_total", "log_monto_mediano", "share_top_proveedores",
            "hhi_proveedores", "n_proveedores", "n_familias"]
feat_ent = VectorAssembler(inputCols=FEAT_ENT, outputCol="features_raw").transform(feat_ent.fillna(0, FEAT_ENT))

- **Uso (Parte II):** KMeans++, DBSCAN/HDBSCAN y jerárquico; tipologías de actores y outliers (PQ1, PQ2)
- Se guardan **sin estandarizar**: la estandarización y el PCA se justifican en la Parte II
- `log1p` en montos y conteos por la cola pesada (`00_EDA/a_entendimiento` §3.f)
- `share_top_*` y `hhi_*` miden dependencia: HHI = Σ (participación de cada contraparte)², 1 = una sola contraparte
- `compite` separa "nunca oferta" de "oferta y nunca gana", que el `fillna(0)` de `tasa_exito` igualaría
- `es_consorcio`: el consorcio es un nodo propio, porque la fuente no publica sus miembros

---
### 2. Matrices de interacción

Formato largo (COO): una fila por celda no vacía, con índices enteros para ALS y SVD, y **una fila por año**

In [8]:
def indexar(df, entrada, salida):
    idx = StringIndexer(inputCol=entrada, outputCol=salida).fit(df)
    return idx, idx.labelsArray[0]

def a_enteros(df, cols):
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("int"))
    return df

ep = (hp.groupBy("entidad_key", "proveedor_key", "anio")
        .agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto")))
ef = (hechos.where("familia <> 'SIN'").groupBy("entidad_key", "familia", "anio")
            .agg(F.countDistinct("ocid").alias("n_procesos"), F.sum("monto_asignado").alias("monto")))
idx_e, lab_e = indexar(ep.select("entidad_key").unionByName(ef.select("entidad_key")), "entidad_key", "entidad_idx")
idx_p, lab_p = indexar(ep, "proveedor_key", "proveedor_idx")
idx_f, lab_f = indexar(ef, "familia", "familia_idx")

In [9]:
m_ep = (a_enteros(idx_p.transform(idx_e.transform(ep)), ["entidad_idx", "proveedor_idx"])
          .withColumn("senal_log_monto", F.log1p(F.coalesce("monto", F.lit(0)))))
m_ef = a_enteros(idx_f.transform(idx_e.transform(ef)), ["entidad_idx", "familia_idx"])
indices = {k: spark.createDataFrame(list(enumerate(v)), ["idx", "clave"])
           for k, v in {"indice_entidad": lab_e, "indice_proveedor": lab_p, "indice_familia": lab_f}.items()}

def densidad(m, fila, columna, n_f, n_c):
    nnz = m.select(fila, columna).distinct().count()
    return nnz, nnz / (n_f * n_c)

n_e, n_p, n_f = len(lab_e), len(lab_p), len(lab_f)
(nnz_ep, d_ep), (nnz_ef, d_ef) = densidad(m_ep, "entidad_idx", "proveedor_idx", n_e, n_p), densidad(m_ef, "entidad_idx", "familia_idx", n_e, n_f)
print(f"entidad × proveedor: {n_e:,} × {n_p:,} | pares {nnz_ep:,} | densidad {d_ep:.4%}")
print(f"entidad × familia  : {n_e:,} × {n_f:,} | pares {nnz_ef:,} | densidad {d_ef:.2%}")

entidad × proveedor: 2,992 × 86,017 | pares 136,099 | densidad 0.0529%
entidad × familia  : 2,992 × 386 | pares 39,348 | densidad 3.41%


- **Uso (Parte III):**
  - `matriz_entidad_proveedor`: ALS implícito y filtrado por vecindario (¿a quién le compra?)
  - `matriz_entidad_familia`: perfil de rubros (¿qué compra?), que sirve para entidades parecidas, *cold start* y para
    distinguir sustituto de rubro nuevo
- **Uso (Parte VI):** SVD truncada de la matriz entidad × proveedor como embedding de entidades y proveedores
- **Señal implícita:** `n_adj` / `n_procesos` (frecuencia) o `senal_log_monto`; la Parte III elige cuál usar como confianza
- **`anio` en cada fila:** el hold-out temporal (entrenar ≤ 2024, evaluar 2025) agrega solo los años de cada lado, **sin
  fuga** de 2025 al entrenamiento
- `StringIndexer` asigna índices de forma distribuida (un `row_number()` sin partición llevaría todo a una sola tarea);
  `indice_*` permite volver del entero a la clave

---
### 3. Grafos

#### a. Bipartito entidad → proveedor

In [10]:
e_p = hp.groupBy("entidad_key", "proveedor_key").agg(F.count("*").alias("n_adj"), F.sum("monto").alias("monto"),
                                                    F.min("anio").alias("anio_min"), F.max("anio").alias("anio_max"))
nodo = lambda df, k, pref, tipo, cols: df.select(F.concat(F.lit(pref), k).alias("id"), F.lit(tipo).alias("tipo"), *cols)
v_ent = nodo(dim_e.join(e_p.select("entidad_key").distinct(), "entidad_key"), "entidad_key", "E|", "entidad", ["nombre", "region"])
v_prov = nodo(dim_p.join(e_p.select("proveedor_key").distinct(), "proveedor_key"), "proveedor_key", "P|", "proveedor",
              ["nombre", "region", "es_consorcio"])
vertices_bip = v_ent.unionByName(v_prov, allowMissingColumns=True)
aristas_bip = e_p.select(F.concat(F.lit("E|"), "entidad_key").alias("src"), F.concat(F.lit("P|"), "proveedor_key").alias("dst"),
                         "n_adj", "monto", "anio_min", "anio_max")

- **Nodos:** entidades (`E|`) y proveedores (`P|`); el prefijo evita fundir una entidad pública que también es proveedor
- **Arista dirigida** entidad → proveedor si hubo al menos una adjudicación; peso `n_adj` o `monto`
- **Uso (Parte IV):** PageRank (GraphFrames) e HITS: hubs = entidades que compran a proveedores centrales, authorities =
  proveedores a los que compran buenos hubs (PQ1)

#### b. Co-participación proveedor — proveedor

In [11]:
MAX_PART, MIN_COMP = 30, 2          # tope ≈ p90 de oferentes por proceso (00_EDA/b_exploracion §5.c)
SHUFFLE = int(spark.conf.get("spark.sql.shuffle.partitions"))

n_part = part.groupBy("ocid").agg(F.count("*").alias("n"))
part_n = (part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
              .repartition(SHUFFLE, "ocid").cache())
pares = (part_n.alias("a").join(part_n.alias("b"),
            (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key")))
         .select(F.col("a.proveedor_key").alias("src"), F.col("b.proveedor_key").alias("dst"), F.col("a.n").alias("n")))
aristas_co = (pares.groupBy("src", "dst").agg(F.count("*").alias("n_coparticipa"), F.sum(1.0 / (F.col("n") - 1)).alias("peso"))
              .where(f"n_coparticipa >= {MIN_COMP}")
              .withColumn("src", F.concat(F.lit("P|"), "src")).withColumn("dst", F.concat(F.lit("P|"), "dst"))).cache()

In [12]:
extremos = aristas_co.select(F.col("src").alias("id")).unionByName(aristas_co.select(F.col("dst").alias("id")))
vertices_co = nodo(dim_p, "proveedor_key", "P|", "proveedor", ["nombre", "region", "es_consorcio"]).join(extremos.distinct(), "id")
grado_co = extremos.groupBy("id").count()
q_grado = grado_co.approxQuantile("count", [.5, .9, .99], .01)
print(f"co-participación: {vertices_co.count():,} nodos | {aristas_co.count():,} aristas | grado p50/p90/p99: {q_grado}")
(grado_co.orderBy(F.desc("count")).limit(5).join(vertices_co.select("id", "nombre"), "id").orderBy(F.desc("count"))
    .show(truncate=50))

co-participación: 60,213 nodos | 1,639,945 aristas | grado p50/p90/p99: [16.0, 113.0, 12059.0]


+-----------------+-----+---------------------------------------+
|               id|count|                                 nombre|
+-----------------+-----+---------------------------------------+
|P|RUC:20611499591|12059|              TVT QONTRATISTAS E.I.R.L.|
|P|RUC:20605753907| 9522|INVERSIONES GENERALES LAURA ROCA S.A.C.|
|P|RUC:20542513439| 7854|      CRISAMYR SOCIEDAD ANONIMA CERRADA|
|P|RUC:20612573621| 7168|               CONSTRUCAPRIGLE E.I.R.L.|
|P|RUC:20542059291| 6523|         CORPORACION GRUPO IBIZA S.A.C.|
+-----------------+-----+---------------------------------------+



- **Criterio de arista:** dos proveedores se conectan si participaron (ofertaron o ganaron) juntos en **≥ 2 procesos** de
  entre 2 y `MAX_PART` participantes
- **Peso** = Σ 1/(n−1) (ponderación de Newman): coincidir en un proceso de 3 pesa 0.5 y en uno de 30, 0.03
- **Tope y umbral:** un proceso de n participantes genera n(n−1)/2 pares, así que los masivos dominarían el grafo con
  vínculos débiles; una sola coincidencia es mayormente casualidad
- **Uso (Partes V y VI):** comunidades (Girvan-Newman/espectral en subgrafo, BigCLAM) y Node2Vec (PQ2)
- **Alternativa (grafo KNN):** si la co-participación fuese escasa, cada proveedor se conectaría a sus k vecinos más
  cercanos en `features_proveedor` estandarizado. Aquí no hace falta (1.6M aristas entre 60k proveedores); queda para la Parte VI como comparación

---
### 4. Particionamiento y plan de ejecución

#### a. Self-join de co-participación

In [13]:
def tiempo(df):
    t0 = time.time(); n = df.count(); return n, round(time.time() - t0, 1)

part_sin = part.select("ocid", "proveedor_key").join(n_part.where(f"n BETWEEN 2 AND {MAX_PART}"), "ocid")
pares_sin = (part_sin.alias("a").join(part_sin.alias("b"),
                (F.col("a.ocid") == F.col("b.ocid")) & (F.col("a.proveedor_key") < F.col("b.proveedor_key"))))
pares_sin.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [ocid#80], [ocid#3940], Inner, (proveedor_key#81 < proveedor_key#3941)
   :- Project [ocid#80, proveedor_key#81, n#2536L]
   :  +- SortMergeJoin [ocid#80], [ocid#3930], Inner
   :     :- Sort [ocid#80 ASC NULLS FIRST], false, 0
   :     :  +- Exchange hashpartitioning(ocid#80, 24), ENSURE_REQUIREMENTS, [plan_id=2425]
   :     :     +- Project [ocid#80, proveedor_key#81]
   :     :        +- Filter (isnotnull(ocid#80) AND isnotnull(proveedor_key#81))
   :     :           +- FileScan parquet [ocid#80,proveedor_key#81,anio#84,mes#85] Batched: true, DataFilters: [isnotnull(ocid#80), isnotnull(proveedor_key#81)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/participantes], PartitionFilters: [anio#84 IN (2023,2024,2025)], PushedFilters: [IsNotNull(ocid), IsNotNull(proveedor_key)], ReadSchema: struct<ocid:string,proveedor_key:string>
   :     +- Sort [ocid#3930 ASC NULLS FIRST],

In [14]:
pares.explain()
print("sin repartition:", tiempo(pares_sin), "| con repartition(ocid) + cache:", tiempo(pares))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [proveedor_key#81 AS src#2595, proveedor_key#2569 AS dst#2596, n#2536L]
   +- SortMergeJoin [ocid#80], [ocid#2568], Inner, (proveedor_key#81 < proveedor_key#2569)
      :- Filter (isnotnull(ocid#80) AND isnotnull(proveedor_key#81))
      :  +- InMemoryTableScan [ocid#80, proveedor_key#81, n#2536L], [isnotnull(ocid#80), isnotnull(proveedor_key#81)]
      :        +- InMemoryRelation [ocid#80, proveedor_key#81, n#2536L], StorageLevel(disk, memory, deserialized, 1 replicas)
      :              +- *(5) Project [ocid#80, proveedor_key#81, n#2536L]
      :                 +- *(5) SortMergeJoin [ocid#80], [ocid#2541], Inner
      :                    :- *(2) Sort [ocid#80 ASC NULLS FIRST], false, 0
      :                    :  +- Exchange hashpartitioning(ocid#80, 24), ENSURE_REQUIREMENTS, [plan_id=2469]
      :                    :     +- *(1) Project [ocid#80, proveedor_key#81]
      :                    :        +- *(1) F

sin repartition: (13092446, 8.4) | con repartition(ocid) + cache: (13092446, 0.9)


- **Sin reparticionar:** cada lado del self-join vuelve a leer `participantes` desde disco y recalcula el conteo por
  proceso (`FileScan` + `Exchange hashpartitioning(ocid)` en cada rama); AQE termina eligiendo `BroadcastHashJoin`
- **Con `repartition(ocid)` + `cache`:** ambos lados leen `InMemoryTableScan` ya particionado por `ocid` y el
  `SortMergeJoin` no necesita otro shuffle
- **Tiempo:** 6.4 s → 0.8 s para los mismos 13.1M pares. Parte de la mejora es el cache (los datos ya están en memoria),
  pero el plan muestra que se elimina el shuffle repetido

#### b. Broadcast y partition pruning

In [15]:
hechos.join(F.broadcast(dim_e), "entidad_key").explain()
leer(final_dir, "hechos").where("anio = 2025 AND mes = 3").select("ocid").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [entidad_key#3, ocid#0, award_id#1, fecha#2, proveedor_key#4, region#5, categoria#6, metodo_detalle#7, postor_unico#8, es_directa#9, n_postores#10, n_proveedores_adj#11L, tiene_contrato#12, monto_adj#13, monto_asignado#14, familia#15, segmento#16, anio#17, mes#18, nombre#94, ruc#95, region#96]
   +- BroadcastHashJoin [entidad_key#3], [entidad_key#93], Inner, BuildRight, false
      :- Filter isnotnull(entidad_key#3)
      :  +- FileScan parquet [ocid#0,award_id#1,fecha#2,entidad_key#3,proveedor_key#4,region#5,categoria#6,metodo_detalle#7,postor_unico#8,es_directa#9,n_postores#10,n_proveedores_adj#11L,tiene_contrato#12,monto_adj#13,monto_asignado#14,familia#15,segmento#16,anio#17,mes#18] Batched: true, DataFilters: [isnotnull(entidad_key#3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [anio#17 IN (2023,2024,2025)], PushedFilters: [IsNotNull(entidad_k

== Physical Plan ==
*(1) Project [ocid#4248]
+- *(1) ColumnarToRow
   +- FileScan parquet [ocid#4248,anio#4265,mes#4266] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/final/hechos], PartitionFilters: [isnotnull(anio#4265), isnotnull(mes#4266), (anio#4265 = 2025), (mes#4266 = 3)], PushedFilters: [], ReadSchema: struct<ocid:string>




- **Dimensión chica (entidades):** `BroadcastHashJoin`. La dimensión se copia a cada executor y la tabla de hechos no se
  mueve (sin `Exchange`)
- **Partition pruning:** el filtro `anio = 2025 AND mes = 3` aparece en `PartitionFilters`: Spark lee solo esa carpeta de
  `hechos` (1 de 36 meses), sin escanear el resto. Es lo que hace barato el hold-out temporal

#### c. Sesgo

In [16]:
(aristas_co.groupBy(F.spark_partition_id().alias("particion")).count()
    .agg(F.min("count").alias("min"), F.expr("percentile_approx(count, 0.5)").alias("mediana"), F.max("count").alias("max")).show())

+-----+-------+-----+
|  min|mediana|  max|
+-----+-------+-----+
|67766|  68316|68907|
+-----+-------+-----+



- Las aristas de co-participación quedan **balanceadas** entre particiones (67,766 a 68,907 filas): el hash sobre
  `(src, dst)` reparte bien
- El sesgo está en los **nodos**, no en las particiones: unos pocos proveedores tienen grado > 12,000 (constructoras que
  participan en miles de procesos). La Parte V deberá tratarlos (filtrar hubs o normalizar) antes de buscar comunidades
- AQE (activo en Spark 3.5) divide particiones sesgadas en los joins; se puede verificar en `localhost:4040`

---
### 5. Exportación de artefactos (`data/artefactos/`)

In [17]:
artefactos = {
    "features_proveedor": feat_prov, "features_entidad": feat_ent,
    "matriz_entidad_proveedor": m_ep, "matriz_entidad_familia": m_ef,
    **{k: v.coalesce(1) for k, v in indices.items()},
    "grafo_bipartito/vertices": vertices_bip, "grafo_bipartito/aristas": aristas_bip,
    "grafo_coparticipacion/vertices": vertices_co, "grafo_coparticipacion/aristas": aristas_co}
for nombre, df in artefactos.items():
    df.write.mode("overwrite").parquet(str(artef_dir / nombre))
print("artefactos:", ", ".join(artefactos))

artefactos: features_proveedor, features_entidad, matriz_entidad_proveedor, matriz_entidad_familia, indice_entidad, indice_proveedor, indice_familia, grafo_bipartito/vertices, grafo_bipartito/aristas, grafo_coparticipacion/vertices, grafo_coparticipacion/aristas


| Artefacto | Grano | Parte | Técnica | Pregunta |
|---|---|---|---|---|
| `features_proveedor` | proveedor | II | KMeans++, DBSCAN/HDBSCAN, jerárquico | PQ1, PQ2 |
| `features_entidad` | entidad | II | ídem; entidades cautivas | PQ1 |
| `matriz_entidad_proveedor` | entidad × proveedor × año | III, VI | ALS implícito, vecindario, SVD | PQ3 |
| `matriz_entidad_familia` | entidad × familia × año | III | perfil de rubros, *cold start* | PQ3 |
| `indice_*` | índice → clave | III, VI | traducir resultados | — |
| `grafo_bipartito` | entidad → proveedor | IV | PageRank, HITS | PQ1 |
| `grafo_coparticipacion` | proveedor — proveedor | V, VI | comunidades, BigCLAM, Node2Vec | PQ2 |

---
### 6. Escritura de resultados

In [18]:
resumen = pd.DataFrame([
    ("proveedores (features)", feat_prov.count()), ("entidades (features)", feat_ent.count()),
    ("pares entidad × proveedor", nnz_ep), ("densidad entidad × proveedor", d_ep),
    ("pares entidad × familia", nnz_ef), ("densidad entidad × familia", d_ef),
    ("vértices bipartito", vertices_bip.count()), ("aristas bipartito", aristas_bip.count()),
    ("vértices co-participación", vertices_co.count()), ("aristas co-participación", aristas_co.count())],
    columns=["medida", "valor"])
escritos = guardar_tablas({"resumen_artefactos": resumen,
                           "grado_coparticipacion": grado_co.groupBy(F.col("count").alias("grado")).agg(F.count("*").alias("nodos"))})
resumen.assign(valor=resumen.valor.map(lambda v: f"{v:.4%}" if v < 1 else f"{v:,.0f}"))

Escrito en 01_Trans/temp/c_uso/: resumen_artefactos, grado_coparticipacion


,medida,valor
0,proveedores (features),"86,017"
1,entidades (features),"2,996"
2,pares entidad × proveedor,"136,099"
3,densidad entidad × proveedor,0.0529%
4,pares entidad × familia,"39,348"
5,densidad entidad × familia,3.4070%
6,vértices bipartito,"89,009"
7,aristas bipartito,"136,099"
8,vértices co-participación,"60,213"
9,aristas co-participación,"1,639,945"


#### a. Verificación

In [19]:
verificar_salidas(escritos)

OK: 2 tablas (parquet + json)


---
### 7. Cerrar sesión

In [20]:
spark.stop()